# 04 - Preview Analytics
**Objetivos:**  
Realizar análises prévias no dataset limpo.  

**Análises prévias:**
- Distribuição dos status
- Distribuição dos tipos de pagamento
- Comparação: Status x Tipo de pagament o
- Comparação: Status x Paciente
- Comparação: Paciente x Ano

In [ ]:
# CÉLULA 1 - CARREGAMENTO

import pandas as pd

df = pd.read_csv("../data/processed/appointments_feature.csv", parse_dates=['appointment_date'])
df.head()

### DISTRIBUIÇÃO POR STATUS
Essa análise tem como objetivo contar o status dos atendimentos, quantos foram atendidos e quantos faltaram e mostrar os valores em porcetagem.

In [164]:
status_count = df['appointment_status'].value_counts().reset_index(name='count')

status_count['proportion'] = status_count['count']\
                            .div(status_count['count'].sum())\
                            .mul(100)\
                            .round(2)

status_count.sort_values('count', ascending=False)
status_count.style.format({'proportion': '{:.2f}%'})                        



,appointment_status,count,proportion
0,Attended,920,81.27%
1,Missed,212,18.73%


### DISTRIBUIÇÃO POR TIPO DE PAGAMENTO
Essa análise tem por objetivo verificar a quantidade de atendimentos que foram usando o método de pagamento 'Por sessão' e quantos utilizaram o método de pagamento 'Pacote mensal' e a porcentagem de cada um em relação ao total.

In [37]:
payment_count = df['payment_type'].value_counts().reset_index(name='count')

payment_count['proportion'] = payment_count['count']\
                            .div(payment_count['count'].sum())\
                            .mul(100)\
                            .round(2)

payment_count.sort_values('count', ascending=False)
payment_count.style.format({'proportion': '{:.2f}%'})

,payment_type,count,proportion
0,PerSession,603,53.27%
1,MonthlyPackage,529,46.73%


### STATUS X TIPO DE PAGAMENTO
Essa análise tem como objetivo comparar a quantidade de presença dos pacientes em relação aos métodos de pagamentos.  

Os resultados mostram que, ao mudar o método de pagamento de 'Por Sessão' para 'Pacote Mensal' houve uma redução nas faltas de quase 50%.

In [133]:
comparacao_pacote = pd.crosstab(df['payment_type'],\
                                df['appointment_status'],\
                                normalize='index')

comparacao_pacote = comparacao_pacote.reset_index().style.format({'Attended': '{:.2%}',
                                                                  'Missed': '{:.2%}'})

comparacao_pacote

appointment_status,payment_type,Attended,Missed
0,MonthlyPackage,87.52%,12.48%
1,PerSession,75.79%,24.21%


### PACIENTE X STATUS
Essa análise tem como objetivo verificar a porcentagem de presença e falta de cada paciente.  

A análise mostra que a maioria dos pacientes possuem uma frequência superior a 80%, e que houve um paciente com taxa de frequência muito baixa, podendo ser classificado como outlier.

In [134]:
attended_percent = pd.crosstab(df['patient_id'],\
                                df['appointment_status'],\
                                normalize='index')\
                                .sort_values('Attended', ascending=False)

attended_percent = attended_percent.reset_index().style.format({'Attended': '{:.2%}',
                                                                'Missed': '{:.2%}'})
attended_percent

appointment_status,patient_id,Attended,Missed
0,P007,94.23%,5.77%
1,P002,92.31%,7.69%
2,P006,91.45%,8.55%
3,P004,89.42%,10.58%
4,P008,89.42%,10.58%
5,P011,88.46%,11.54%
6,P012,83.33%,16.67%
7,P010,82.20%,17.80%
8,P005,73.56%,26.44%
9,P003,73.08%,26.92%


### PACIENTE X ANO
Essa análise tem como objetivo verificar a taxa de faltas dos pacientes em 2022 e em 2023 e calcular qual foi a taxa de melhoria de um ano para o outro com a adição do pacote mensal. 

Os resultados mostram uma taxa de melhora de em 6 dos 7 pacientes com dados dos dois anos, e que os novos pacientes que entraram já no pacote, demonstram taxa de falta semelhante aos valores dos pacientes que já estavam em atendimento.

Obs.: O paciente P004 teve sua taxa de melhoria negativa, devido a um ano difícil em relação a sua saúde.

In [162]:
comparacao_anual = df.groupby(['patient_id',
                               df['appointment_date'].dt.year])['appointment_status']\
                                .apply(lambda x: (x == 'Missed').mean())\
                                .unstack()
                                
comparacao_anual.columns = ['miss_rate_22','miss_rate_23']

comparacao_anual['melhora'] = comparacao_anual['miss_rate_22'] - comparacao_anual['miss_rate_23']

comparacao_anual.sort_values('melhora', ascending=False)\
                .reset_index()\
                .style.format({'miss_rate_22': '{:.2%}',
                               'miss_rate_23': '{:.2%}',
                               'melhora': '{:.2%}'},
                                na_rep="-")

,patient_id,miss_rate_22,miss_rate_23,melhora
0,P005,35.58%,17.31%,18.27%
1,P010,26.42%,10.77%,15.65%
2,P009,34.78%,19.23%,15.55%
3,P008,17.31%,3.85%,13.46%
4,P007,9.62%,1.92%,7.69%
5,P006,10.77%,5.77%,5.00%
6,P004,0.00%,21.15%,-21.15%
7,P001,61.54%,-,-
8,P002,7.69%,-,-
9,P003,26.92%,-,-
